In [6]:
pip install easyocr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.8/422.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [7]:
import easyocr
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [29]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import pandas as pd
from PIL import Image

# Define CRNN
class CRNN(nn.Module):
    def __init__(self, img_h, num_classes):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 512, 3, 1, 1), nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.Conv2d(512, 512, 3, 1, 1), nn.ReLU(),
            nn.BatchNorm2d(512), nn.MaxPool2d((2, 1))
        )
        self.rnn1 = nn.LSTM(1024, 256, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        conv = self.cnn(x)  # (B, C, H, W)
        b, c, h, w = conv.size()
        conv = conv.permute(0, 3, 1, 2)  # (B, W, C, H)
        conv = conv.reshape(b, w, c * h) # (B, W, C*H)
        rnn_out, _ = self.rnn1(conv)
        rnn_out, _ = self.rnn2(rnn_out)
        output = self.fc(rnn_out)       # (B, W, num_classes)
        return output.permute(1, 0, 2)  # (W, B, num_classes) for CTC

# Custom Dataset
class OCRDataset(Dataset):
    def __init__(self, csv_path, img_dir, charset, transform=None):
        self.data = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform
        self.charset = charset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['filename'])).convert('L')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor([self.charset.index(c) for c in row['label']])
        return img, label, len(label)

# Character set and config
charset = list("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZOF")
num_classes = len(charset) + 1  # CTC blank

# Transforms
transform = transforms.Compose([
    transforms.Resize((32, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Dataset
train_dataset = OCRDataset("labels.csv", "images", charset, transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: x)

# Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(img_h=32, num_classes=num_classes).to(device)
criterion = nn.CTCLoss(blank=num_classes - 1, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop (simplified)
for epoch in range(70):
    model.train()
    for batch in train_loader:
        imgs, labels, label_lens = zip(*batch)
        imgs = torch.stack(imgs).to(device)
        targets = torch.cat(labels).to(device)
        input_lengths = torch.full(size=(imgs.size(0),), fill_value=imgs.size(-1) // 4, dtype=torch.long)
        target_lengths = torch.tensor(label_lens, dtype=torch.long)

        output = model(imgs)
        loss = criterion(output, targets, input_lengths, target_lengths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

torch.save(model.state_dict(), "crnn_ocr.pth")




Epoch 1, Loss: 4.2810
Epoch 2, Loss: 2.5875
Epoch 3, Loss: 2.7028
Epoch 4, Loss: 2.0768
Epoch 5, Loss: 2.2866
Epoch 6, Loss: 3.1064
Epoch 7, Loss: 3.1662
Epoch 8, Loss: 2.2301
Epoch 9, Loss: 3.1540
Epoch 10, Loss: 2.2779
Epoch 11, Loss: 2.1760
Epoch 12, Loss: 1.6370
Epoch 13, Loss: 2.5334
Epoch 14, Loss: 1.9300
Epoch 15, Loss: 4.1625
Epoch 16, Loss: 2.5604
Epoch 17, Loss: 1.9642
Epoch 18, Loss: 1.9709
Epoch 19, Loss: 2.8151
Epoch 20, Loss: 3.6762
Epoch 21, Loss: 2.0161
Epoch 22, Loss: 1.0054
Epoch 23, Loss: 0.7846
Epoch 24, Loss: 2.4258
Epoch 25, Loss: 1.1753
Epoch 26, Loss: 1.2991
Epoch 27, Loss: 0.9179
Epoch 28, Loss: 0.3398
Epoch 29, Loss: 0.4398
Epoch 30, Loss: 0.2357
Epoch 31, Loss: 0.7573
Epoch 32, Loss: -1.2552
Epoch 33, Loss: 0.6473
Epoch 34, Loss: 0.8257
Epoch 35, Loss: 0.6574
Epoch 36, Loss: -0.0573
Epoch 37, Loss: 0.2298
Epoch 38, Loss: 1.0428
Epoch 39, Loss: 0.0237
Epoch 40, Loss: -0.2629
Epoch 41, Loss: 1.2018
Epoch 42, Loss: 2.2216
Epoch 43, Loss: 0.5119
Epoch 44, Loss: 0

In [36]:
import os
from PIL import Image, ImageDraw, ImageFont
import pandas as pd

# 샘플로 만들 문자열들 (근무 타입)
labels = ["D", "E", "N", "OF"] * 50  # 200장 생성
save_dir = "images"
os.makedirs(save_dir, exist_ok=True)

# 폰트 설정 (Colab이면 /usr/share/fonts/truetype/nanum/NanumGothic.ttf 추천)
try:
    font = ImageFont.truetype("arial.ttf", 40)
except:
    font = ImageFont.load_default()

# 이미지 생성
records = []
for idx, label in enumerate(labels):
    img = Image.new("L", (128, 32), color=255)  # 흑백, 흰 배경
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0, 0), label, font=font)
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    draw.text(((128 - w) / 2, (32 - h) / 2), label, font=font, fill=0)
    fname = f"sample_{idx:04d}.png"
    img.save(os.path.join(save_dir, fname))
    records.append((fname, label))

# 라벨 CSV 저장
df = pd.DataFrame(records, columns=["filename", "label"])
df.to_csv("labels.csv", index=False)

print(f"{len(records)} images generated to '{save_dir}' and labels.csv created.")


200 images generated to 'images' and labels.csv created.


In [60]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

# 문자셋과 모델 구조 동일해야 함
charset = list("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZOF")
num_classes = len(charset) + 1  # CTC blank

# 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((32, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 모델 로딩 (학습 때와 동일한 CRNN 구조 사용해야 함)
class CRNN(torch.nn.Module):
    def __init__(self, img_h, num_classes):
        super(CRNN, self).__init__()
        self.cnn = torch.nn.Sequential(
            torch.nn.Conv2d(1, 64, 3, 1, 1), torch.nn.ReLU(), torch.nn.MaxPool2d(2, 2),
            torch.nn.Conv2d(64, 128, 3, 1, 1), torch.nn.ReLU(), torch.nn.MaxPool2d(2, 2),
            torch.nn.Conv2d(128, 256, 3, 1, 1), torch.nn.ReLU(),
            torch.nn.Conv2d(256, 256, 3, 1, 1), torch.nn.ReLU(), torch.nn.MaxPool2d((2, 1)),
            torch.nn.Conv2d(256, 512, 3, 1, 1), torch.nn.ReLU(),
            torch.nn.BatchNorm2d(512),
            torch.nn.Conv2d(512, 512, 3, 1, 1), torch.nn.ReLU(),
            torch.nn.BatchNorm2d(512), torch.nn.MaxPool2d((2, 1))
        )
        self.rnn1 = torch.nn.LSTM(1024, 256, bidirectional=True, batch_first=True)
        self.rnn2 = torch.nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        self.fc = torch.nn.Linear(512, num_classes)

    def forward(self, x):
        conv = self.cnn(x)
        b, c, h, w = conv.size()
        conv = conv.permute(0, 3, 1, 2).reshape(b, w, c * h)
        rnn_out, _ = self.rnn1(conv)
        rnn_out, _ = self.rnn2(rnn_out)
        output = self.fc(rnn_out)
        return output.permute(1, 0, 2)

# 예측 함수
def predict_text(model, image_path):
    model.eval()
    image = Image.open(image_path).convert('L')
    image = transform(image).unsqueeze(0).to(next(model.parameters()).device)
    with torch.no_grad():
        output = model(image)
        output = F.log_softmax(output, dim=2)
        _, preds = output.max(2)
        preds = preds.transpose(1, 0).contiguous().view(-1)
        # CTC decoding
        decoded = []
        prev = -1
        for p in preds:
            if p.item() != prev and p.item() != num_classes - 1:
                decoded.append(charset[p.item()])
            prev = p.item()
        return ''.join(decoded)

# 사용 예시
model = CRNN(32, num_classes)
model.load_state_dict(torch.load("crnn_ocr.pth"))
model.to("cuda" or "cpu")
print(predict_text(model, "sample_data/cell_d.png"))


N


In [48]:
import pandas as pd

df = pd.read_csv("labels.csv")
print(df.head(10))  # 상위 10개만 미리 보기



          filename label
0  sample_0000.png     D
1  sample_0001.png     E
2  sample_0002.png     N
3  sample_0003.png    OF
4  sample_0004.png     D
5  sample_0005.png     E
6  sample_0006.png     N
7  sample_0007.png    OF
8  sample_0008.png     D
9  sample_0009.png     E
